# 📊 Speech Enhancement — Step 4: Evaluate Performance

This notebook:
1. Mounts Google Drive and loads the trained U-Net model.
2. **Denoises** the QC audio using `denoise_audio()` from `predict.py` (same as notebook 03).
3. Measures how well the model denoises speech using three standard metrics.

| Metric | What it measures | Range |
|--------|-----------------|-------|
| **SNR** | Signal-to-noise ratio improvement | Higher = better (dB) |
| **PESQ** | Perceptual speech quality (ITU standard) | –0.5 → 4.5, higher = better |
| **STOI** | Speech intelligibility | 0 → 1, higher = better |

### Prerequisites
- Notebooks 01 and 02 completed.
- Drive contains `data/sounds/` QC pairs (clean + noisy) from `create_data()`.
- Drive contains `weights/model_unet.weights.h5` from training.

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
%pip install -q librosa soundfile pesq pystoi

In [ ]:
# ── 3. Clone / pull repo & add src/ to path ────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/theweird-kid/speech_enhancement.git'  # ← update
REPO_DIR = '/content/speech_enhancement'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready ✓')

In [ ]:
# ── 4. Config & imports ────────────────────────────────────────────────────
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import pandas as pd
from pesq import pesq
from pystoi import stoi
import config as C

SR = C.SAMPLE_RATE   # 8000 Hz
print(f'Sample rate: {SR} Hz')
print(f'Sounds dir : {C.SOUND_DIR}')
print(f'Pred dir   : {C.PRED_DIR}')

In [ ]:
# ── 5. Helper functions ────────────────────────────────────────────────────

def load_wav(path, sr=SR):
    """Load a WAV and return a float32 array resampled to `sr`."""
    y, _ = librosa.load(path, sr=sr)
    return y.astype(np.float32)

def align(a, b):
    """Trim both arrays to the same length."""
    n = min(len(a), len(b))
    return a[:n], b[:n]

def compute_snr(clean, enhanced):
    noise = clean - enhanced
    return 10 * np.log10(np.sum(clean**2) / (np.sum(noise**2) + 1e-8))

def compute_metrics(clean, test, sr=SR):
    """Return dict of SNR, PESQ, STOI for a (clean, test) pair."""
    c, t = align(clean, test)
    return {
        'SNR (dB)': round(compute_snr(c, t), 3),
        'PESQ'    : round(pesq(sr, c, t, 'nb'), 3),
        'STOI'    : round(stoi(c, t, sr, extended=False), 3),
    }

print('Helpers defined ✓')

In [ ]:
# ── 6. Generate denoised audio using predict.py (same as notebook 03) ─────
from predict import denoise_audio

CLEAN_WAV    = os.path.join(C.SOUND_DIR, 'clean_voice_long.wav')
NOISY_WAV    = os.path.join(C.SOUND_DIR, 'noisy_voice_long.wav')
DENOISED_WAV = os.path.join(C.PRED_DIR,  'denoised_noisy_voice_long.wav')
WEIGHTS      = C.PRETRAINED_WEIGHTS

os.makedirs(C.PRED_DIR, exist_ok=True)

print(f'Noisy input  : {NOISY_WAV}')
print(f'Denoised out : {DENOISED_WAV}')
print(f'Weights      : {WEIGHTS}')
print()

# Run denoising  (this is the same function notebook 03 calls)
_ = denoise_audio(
    input_path   = NOISY_WAV,
    output_path  = DENOISED_WAV,
    weights_path = WEIGHTS,
)
print('\nDenoised audio generated ✓')

In [ ]:
# ── 7. Load audio & compute before / after metrics ─────────────────────────
clean_full    = load_wav(CLEAN_WAV)
noisy_full    = load_wav(NOISY_WAV)
denoised_full = load_wav(DENOISED_WAV)

# PESQ/STOI are slow on very long audio — evaluate on first 30 seconds
MAX_SECONDS = 30
MAX_SAMPLES = MAX_SECONDS * SR

clean_a    = clean_full[:MAX_SAMPLES]
noisy_a    = noisy_full[:MAX_SAMPLES]
denoised_a = denoised_full[:MAX_SAMPLES]

print(f'Full audio   : {len(clean_full)/SR:.1f} s')
print(f'Denoised     : {len(denoised_full)/SR:.1f} s')
print(f'Evaluating on first {MAX_SECONDS}s ({MAX_SAMPLES} samples)')

print('\n=== Noisy vs Clean (before enhancement) ===')
before = compute_metrics(clean_a, noisy_a)
for k, v in before.items():
    print(f'  {k}: {v}')

print('\n=== Denoised vs Clean (after enhancement) ===')
after = compute_metrics(clean_a, denoised_a)
for k, v in after.items():
    print(f'  {k}: {v}')

# Summary table
metrics = ['SNR (dB)', 'PESQ', 'STOI']
df = pd.DataFrame({
    'Metric':      metrics,
    'Before':      [before[m] for m in metrics],
    'After':       [after[m]  for m in metrics],
})
df['Improvement'] = (df['After'] - df['Before']).round(3)
print('\n', df.to_string(index=False))

In [ ]:
# ── 8. Bar chart: before vs after ──────────────────────────────────────────
metrics     = ['SNR (dB)', 'PESQ', 'STOI']
before_vals = [before[m] for m in metrics]
after_vals  = [after[m]  for m in metrics]

x, width = np.arange(len(metrics)), 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, before_vals, width, label='Noisy (before)', color='#e05c5c')
bars2 = ax.bar(x + width/2, after_vals,  width, label='Denoised (after)', color='#4caf7d')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Speech Enhancement — Before vs After', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.savefig(os.path.join(C.WEIGHTS_DIR, 'performance_chart.png'), dpi=120)
plt.show()

# Summary table
df_summary = pd.DataFrame({
    'Metric': metrics,
    'Before': before_vals,
    'After':  after_vals,
})
df_summary['Improvement'] = (df_summary['After'] - df_summary['Before']).round(3)
print(df_summary.to_string(index=False))

In [ ]:
# ── 9. Spectrogram comparison: noisy vs denoised ──────────────────────────
import librosa.display

def load_spec(wav_path, sr=SR):
    y, _ = librosa.load(wav_path, sr=sr)
    S    = librosa.stft(y, n_fft=C.N_FFT, hop_length=C.HOP_LENGTH_FFT)
    return librosa.amplitude_to_db(np.abs(S), ref=np.max)

S_noisy    = load_spec(NOISY_WAV)
S_denoised = load_spec(DENOISED_WAV)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
librosa.display.specshow(S_noisy,   sr=SR,
                         hop_length=C.HOP_LENGTH_FFT,
                         x_axis='time', y_axis='hz', ax=axes[0], cmap='magma')
axes[0].set_title('Noisy Voice Spectrogram')

librosa.display.specshow(S_denoised, sr=SR,
                         hop_length=C.HOP_LENGTH_FFT,
                         x_axis='time', y_axis='hz', ax=axes[1], cmap='magma')
axes[1].set_title('Denoised Voice Spectrogram')

plt.tight_layout()
plt.show()